In [1]:
import pandas as pd
import sklearn 
import numpy as np


Creating Dummy Features

In [2]:
crash_roads_data = pd.read_csv('crash_data_roads.csv')
crash_parties_background_data = pd.read_csv('crash_parties_background_info.csv')
crash_parties_car_data = pd.read_csv('crash_parties_car_data.csv')
crash_parties_speed_limit_data = pd.read_csv('crash_parties_speed_limit_data.csv')
crash_data_freeway = pd.read_csv('freeway_crashes.csv') #This and local might be redundant
crash_data_local = pd.read_csv('local_crashes.csv')
crash_data_collision_factors = pd.read_csv('main_collision_factors.csv')
crash_roads_speed_limit_data = pd.merge(crash_roads_data, crash_parties_speed_limit_data, on='CollisionId', how='inner')
crash_parties_car_background_data = pd.merge(crash_parties_car_data, crash_parties_background_data, on='CollisionId', how='inner')
crash_parties_car_collision_background = pd.merge(crash_parties_car_background_data, crash_data_collision_factors, on='CollisionId', how='inner')



In [3]:
crash_parties_background_data.head()

,CollisionId,PartyType,Special Information,IsAtFault
0,4541902,Driver,CELL PHONE USE UNKNOWN,True
1,4541901,Driver,CELL PHONE NOT IN USE,False
2,4541901,Driver,CELL PHONE USE UNKNOWN,True
3,4541900,Driver,CELL PHONE NOT IN USE,False
4,4541900,Driver,CELL PHONE NOT IN USE,False


In [4]:
crash_parties_car_collision_background.head()

,CollisionId,Vehicle1Year,Vehicle1Make,Vehicle1Model,Vehicle1Color,V1IsVehicleTowed,PartyType,Special Information,IsAtFault,IsTowAway,MotorVehicleInvolvedWithDesc,NumberInjured,NumberKilled,Primary Collision Factor Code,Primary Collision Factor Violation,PrimaryCollisionFactorIsCited,PrimaryCollisionPartyNumber,parsed_values
0,4541901,2005.0,FORD,F-150,Gray,True,Driver,CELL PHONE NOT IN USE,False,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
1,4541901,2005.0,FORD,F-150,Gray,True,Driver,CELL PHONE USE UNKNOWN,True,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
2,4541901,2002.0,FORD,TAURUS,GLD,True,Driver,CELL PHONE NOT IN USE,False,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
3,4541901,2002.0,FORD,TAURUS,GLD,True,Driver,CELL PHONE USE UNKNOWN,True,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
4,4541899,2019.0,TOYOTA,4-RUNNER,Blue,False,Driver,CELL PHONE NOT IN USE,False,False,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21658(a),False,1.0,21658.0


In [16]:
top_10_models = crash_parties_car_collision_background["Vehicle1Model"].value_counts().head(10).index
crash_parties_car_collision_background = crash_parties_car_collision_background[crash_parties_car_collision_background['Vehicle1Model'].isin(top_10_models)]

In [18]:
top_10_colors = crash_parties_car_collision_background["Vehicle1Color"].value_counts().head(10).index
crash_parties_car_collision_background = crash_parties_car_collision_background[crash_parties_car_collision_background['Vehicle1Color'].isin(top_10_colors)]

In [24]:
crash_parties_car_collision_background[['IsAtFault', 'IsTowAway']] = crash_parties_car_collision_background[['IsAtFault', 'IsTowAway']].astype(int)

In [25]:
crash_parties_car_collision_background

,CollisionId,Vehicle1Year,Vehicle1Make,Vehicle1Model,Vehicle1Color,V1IsVehicleTowed,PartyType,Special Information,IsAtFault,IsTowAway,MotorVehicleInvolvedWithDesc,NumberInjured,NumberKilled,Primary Collision Factor Code,Primary Collision Factor Violation,PrimaryCollisionFactorIsCited,PrimaryCollisionPartyNumber,parsed_values
10,4541896,2012.0,TOYOTA,COROLLA,Black,False,Driver,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,0.0,0.0,A,VC 22350,True,1.0,22350.0
11,4541896,2012.0,TOYOTA,COROLLA,Black,False,Driver,CELL PHONE NOT IN USE,1,0,OTHER MOTOR VEHICLE,0.0,0.0,A,VC 22350,True,1.0,22350.0
12,4541895,2017.0,HONDA,CIVIC,Blue,False,Driver,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,0.0,0.0,A,VC 22106,False,1.0,22106.0
13,4541895,2017.0,HONDA,CIVIC,Blue,False,Driver,CELL PHONE NOT IN USE,1,0,OTHER MOTOR VEHICLE,0.0,0.0,A,VC 22106,False,1.0,22106.0
14,4541895,2016.0,CHEVROLET,SILVERADO,Gray,False,Driver,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,0.0,0.0,A,VC 22106,False,1.0,22106.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1080102,5105674,2015.0,TOYOTA,CAMRY,Silver,NaN,Driver,CELL PHONE NOT IN USE,1,1,PARKED MOTOR VEHICLE,0.0,0.0,A,23152(a),True,1.0,23152.0
1080106,5105672,2019.0,HONDA,CIVIC,Gray,NaN,Driver,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,1.0,0.0,A,22107,False,1.0,22107.0
1080107,5105672,2019.0,HONDA,CIVIC,Gray,NaN,Driver,CELL PHONE NOT IN USE,1,0,OTHER MOTOR VEHICLE,1.0,0.0,A,22107,False,1.0,22107.0
1080108,5105672,2025.0,HONDA,CIVIC,Silver,NaN,Driver,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,1.0,0.0,A,22107,False,1.0,22107.0


In [21]:
crash_parties_car_collision_background_dummies = pd.get_dummies(crash_parties_car_collision_background, columns = ['PartyType'], dtype = int)

In [27]:
crash_parties_car_collision_background_dummies.drop('Primary Collision Factor Violation', axis = 1,  inplace = True)

In [28]:
crash_parties_car_collision_background_dummies.columns

Index(['CollisionId', 'Vehicle1Year', 'Vehicle1Make', 'Vehicle1Model',
       'Vehicle1Color', 'V1IsVehicleTowed', 'Special Information', 'IsAtFault',
       'IsTowAway', 'MotorVehicleInvolvedWithDesc', 'NumberInjured',
       'NumberKilled', 'Primary Collision Factor Code',
       'PrimaryCollisionFactorIsCited', 'PrimaryCollisionPartyNumber',
       'parsed_values', 'PartyType_Bicyclist', 'PartyType_Driver',
       'PartyType_Operator', 'PartyType_Other', 'PartyType_ParkedVehicle',
       'PartyType_Pedestrian'],
      dtype='str')

In [29]:
crash_parties_car_collision_background_dummies = pd.get_dummies(crash_parties_car_collision_background, columns = ['PartyType', 'parsed_values'], dtype = int)

In [30]:
crash_parties_car_collision_background_dummies.head()

,CollisionId,Vehicle1Year,Vehicle1Make,Vehicle1Model,Vehicle1Color,V1IsVehicleTowed,Special Information,IsAtFault,IsTowAway,MotorVehicleInvolvedWithDesc,...,parsed_values_27465.0,parsed_values_27467.0,parsed_values_28001.0,parsed_values_29003.0,parsed_values_29004.0,parsed_values_35250.0,parsed_values_36509.0,parsed_values_38025.0,parsed_values_38300.0,parsed_values_38305.0
10,4541896,2012.0,TOYOTA,COROLLA,Black,False,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,...,0,0,0,0,0,0,0,0,0,0
11,4541896,2012.0,TOYOTA,COROLLA,Black,False,CELL PHONE NOT IN USE,1,0,OTHER MOTOR VEHICLE,...,0,0,0,0,0,0,0,0,0,0
12,4541895,2017.0,HONDA,CIVIC,Blue,False,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,...,0,0,0,0,0,0,0,0,0,0
13,4541895,2017.0,HONDA,CIVIC,Blue,False,CELL PHONE NOT IN USE,1,0,OTHER MOTOR VEHICLE,...,0,0,0,0,0,0,0,0,0,0
14,4541895,2016.0,CHEVROLET,SILVERADO,Gray,False,CELL PHONE NOT IN USE,0,0,OTHER MOTOR VEHICLE,...,0,0,0,0,0,0,0,0,0,0


Scale Standarization

Since this is model is not really dependent on numeric features, I believe that scale standarization isnt needed here

Split Data

For splitting the data, to make it simple, we should just classify who is more likely to cause an accident rather than who will get injured or killed since most of the time its 0. We need to use groupshufflesplit here since there are many different values related to each other via collisionID we need to use this so that there is no data leaks